In [6]:
import time
import re
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.options import Options
from selenium.common.exceptions import NoSuchElementException, TimeoutException
from bs4 import BeautifulSoup

LIST_URL = "https://youth.europa.eu/go-abroad/volunteering/opportunities_en"
BASE_URL = "https://youth.europa.eu"

chrome_options = Options()
chrome_options.add_argument("--headless")
chrome_options.add_argument("--disable-gpu")
chrome_options.add_argument("window-size=1920x1080")
chrome_options.add_argument("user-agent=Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.114 Safari/537.36")

service = Service(ChromeDriverManager().install())
driver = webdriver.Chrome(service=service, options=chrome_options)

print(f"Connecting to the main listing page with Selenium: {LIST_URL}...")
target_project_count = 0

try:
    driver.get(LIST_URL)
    
    # 1. FIND OUT THE TARGET NUMBER OF PROJECTS
    try:
        target_label_xpath = "//h5/span[contains(text(), 'opportunities found')]"
        WebDriverWait(driver, 15).until(
            EC.presence_of_element_located((By.XPATH, target_label_xpath))
        )
        target_text = driver.find_element(By.XPATH, target_label_xpath).text
        match = re.search(r'(\d+)', target_text)
        if match:
            target_project_count = int(match.group(1))
            print(f"Target number of projects found: {target_project_count}")
        else:
            raise Exception("The target number text was found but the number could not be extracted.")
            
    except Exception as e:
        print(f"ERROR: Total number of projects not found: {e}")
        raise

    # 2. XPATH of the "Load more" button
    load_more_xpath = "//button[contains(text(), 'Load more')]"
    
    # 3. HYBRID LOOP: click UNTIL the target number is reached
    while len(driver.find_elements(By.CLASS_NAME, "card-item")) < target_project_count:
        try:
            current_project_count = len(driver.find_elements(By.CLASS_NAME, "card-item"))
            
            if current_project_count >= target_project_count:
                print(f"All listings loaded (current: {current_project_count} / target: {target_project_count}).")
                break

            load_more_button = WebDriverWait(driver, 10).until(
                EC.element_to_be_clickable((By.XPATH, load_more_xpath))
            )
            driver.execute_script("arguments[0].click();", load_more_button)
            
            print(f"-> 'Load more' clicked. Loaded: {current_project_count}/{target_project_count}")
            time.sleep(2) 
            
        except (NoSuchElementException, TimeoutException):
            current_project_count = len(driver.find_elements(By.CLASS_NAME, "card-item"))
            print(f"All listings loaded (the Load more button disappeared). Loaded: {current_project_count}")
            break
        except Exception as e:
            print(f"An unexpected error occurred while clicking the button: {e}")
            break

    # 4. After all listings are loaded, COLLECT THE LINKS
    print("The whole page is loaded. Collecting links...")
    html_source = driver.page_source
    soup = BeautifulSoup(html_source, "html.parser")
    listing_blocks = soup.find_all("div", class_="card-item")
    
    link_list = []
    for block in listing_blocks:
        link_element = block.find("div", class_="card-action").find("a")
        if link_element and link_element.has_attr('href'):
            link = BASE_URL + link_element['href']
            link_list.append(link)

    # 5. Write the links to a file
    with open("listing_links.txt", "w") as f:
        for link in link_list:
            f.write(link + "\n")

    print(f"\n--- CODE 1 COMPLETED ---")
    print(f"{len(link_list)} project links in total saved to 'listing_links.txt'.")

except Exception as e:
    print(f"ERROR: A major problem occurred while running Selenium: {e}")

finally:
    driver.quit()
    print("Browser closed.")

Connecting to the main listing page with Selenium: https://youth.europa.eu/go-abroad/volunteering/opportunities_en...
Target number of projects found: 951
-> 'Load more' clicked. Loaded: 8/951
-> 'Load more' clicked. Loaded: 16/951
-> 'Load more' clicked. Loaded: 16/951
-> 'Load more' clicked. Loaded: 32/951
-> 'Load more' clicked. Loaded: 40/951
-> 'Load more' clicked. Loaded: 48/951
-> 'Load more' clicked. Loaded: 56/951
-> 'Load more' clicked. Loaded: 56/951
-> 'Load more' clicked. Loaded: 72/951
-> 'Load more' clicked. Loaded: 72/951
-> 'Load more' clicked. Loaded: 80/951
-> 'Load more' clicked. Loaded: 88/951
-> 'Load more' clicked. Loaded: 104/951
-> 'Load more' clicked. Loaded: 112/951
-> 'Load more' clicked. Loaded: 120/951
-> 'Load more' clicked. Loaded: 128/951
-> 'Load more' clicked. Loaded: 136/951
-> 'Load more' clicked. Loaded: 144/951
-> 'Load more' clicked. Loaded: 152/951
-> 'Load more' clicked. Loaded: 160/951
-> 'Load more' clicked. Loaded: 168/951
-> 'Load more' cli

In [7]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time

BASE_URL = "https://youth.europa.eu"

# -------------------------------------------------------------------------
# DETAIL PAGE ANALYSIS FUNCTION
# (429 error handling and retry logic added)
# -------------------------------------------------------------------------
def extract_project_description_robust(detail_url, retries=1):
    """
    Goes to the URL; on a 429 error it waits 10 seconds and tries again.
    """
    try:
        page = requests.get(detail_url)
        page.raise_for_status() # Raise an exception on error (including 429)
        
        soup = BeautifulSoup(page.content, "html.parser")
        content_cards = soup.find_all("div", class_="card-content")
        full_description_text = ""
        
        for card in content_cards:
            h6_header = card.find("h6")
            if h6_header and "Description" in h6_header.text:
                all_paragraphs = card.find_all("p")
                for p in all_paragraphs:
                    full_description_text += p.text.strip() + "\n\n" 
                break 
        
        return full_description_text.strip() if full_description_text else "DESCRIPTION NOT FOUND"
            
    except requests.exceptions.HTTPError as e:
        if e.response.status_code == 429 and retries > 0:
            print("    -> 429 error! The server blocked the request. Waiting 10 seconds before retrying...")
            time.sleep(10) # A LONG wait to get past the block
            # Call the function again with one fewer remaining attempt
            return extract_project_description_robust(detail_url, retries - 1)
        else:
            return f"ERROR: {e.response.status_code} error"
    except requests.exceptions.RequestException as e:
        return "ERROR: Connection error"

# -------------------------------------------------------------------------
# MAIN PROCESSING LOOP (CODE 2)
# -------------------------------------------------------------------------
print("--- CODE 2 STARTED: Fetching details ---")

try:
    with open("listing_links.txt", "r") as f:
        link_list = [line.strip() for line in f.readlines()]
    print(f"'{len(link_list)}' links read from the file.")
except FileNotFoundError:
    print("ERROR: 'listing_links.txt' not found. Please run Code 1 first.")
    # The raise command stops this cell
    raise

project_records = []
failed_links = 0

for i, link in enumerate(link_list):
    print(f"Processing... ({i+1}/{len(link_list)}) - {link}")
    
    # Call the main function (1 retry allowed)
    full_description = extract_project_description_robust(link, retries=1)

    if "ERROR" in full_description or "NOT FOUND" in full_description:
        print(f"    -> WARNING: Skipping this listing. (Reason: {full_description[:50]}...)")
        failed_links += 1

    project_records.append({
        "link": link,
        "full_description": full_description
    })
    
    # --- 429 BLOCKER ---
    # Wait 1 second between requests. (This line is very important)
    time.sleep(1) 

# 6. Convert the data to a DataFrame and save it
df_projects = pd.DataFrame(project_records)
# We could merge it with the title etc. in 'youth_portal_listings.csv'
# but for now we only save the descriptions:
df_projects.to_csv("youth_portal_listings_detailed.csv", index=False, encoding='utf-8-sig')

print(f"\n--- CODE 2 COMPLETED ---")
print(f"{len(df_projects)} project listings processed in total.")
if failed_links > 0:
    print(f"WARNING: The detail page of {failed_links} listings COULD NOT BE FETCHED.")
print("All data saved to 'youth_portal_listings_detailed.csv'.")

--- CODE 2 STARTED: Fetching details ---
'952' links read from the file.
Processing... (1/952) - https://youth.europa.eu/solidarity/opportunity/49017_en
Processing... (2/952) - https://youth.europa.eu/solidarity/opportunity/49016_en
Processing... (3/952) - https://youth.europa.eu/solidarity/opportunity/49015_en
Processing... (4/952) - https://youth.europa.eu/solidarity/opportunity/49014_en
Processing... (5/952) - https://youth.europa.eu/solidarity/opportunity/49013_en
Processing... (6/952) - https://youth.europa.eu/solidarity/opportunity/49012_en
Processing... (7/952) - https://youth.europa.eu/solidarity/opportunity/49011_en
Processing... (8/952) - https://youth.europa.eu/solidarity/opportunity/49010_en
Processing... (9/952) - https://youth.europa.eu/solidarity/opportunity/49011_en
Processing... (10/952) - https://youth.europa.eu/solidarity/opportunity/49010_en
Processing... (11/952) - https://youth.europa.eu/solidarity/opportunity/49009_en
Processing... (12/952) - https://youth.europa

In [10]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.options import Options

# -------------------------------------------------------------------------
# STEP 3: EUROPASS CV PARSER (UPDATED WITH SELENIUM)
# -------------------------------------------------------------------------

CV_URL = "https://europa.eu/europass/eportfolio/api/eprofile/shared-profile/example/12345678-1234-1234-1234-123456789012?view=html"

chrome_options = Options()
chrome_options.add_argument("--headless")
chrome_options.add_argument("--disable-gpu")
chrome_options.add_argument("window-size=1920x1080")
chrome_options.add_argument("user-agent=Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.114 Safari/537.36")

service = Service(ChromeDriverManager().install())
driver = webdriver.Chrome(service=service, options=chrome_options)

print(f"IP 1 - Step 3 started: fetching the Europass CV link with Selenium...\nURL: {CV_URL}")

cv_text_list = []
cv_skill_list = [] 

try:
    driver.get(CV_URL)
    
    # Wait for the page to load dynamically (until the Skills heading exists)
    skills_header_xpath = "//h2[contains(text(), 'Skills')]"
    WebDriverWait(driver, 15).until(
        EC.presence_of_element_located((By.XPATH, skills_header_xpath))
    )
    print("Europass profile (JavaScript) loaded. Parsing data...")
    
    html_source = driver.page_source
    soup = BeautifulSoup(html_source, "html.parser")

    # --- 1. Fetch the Education and training section ---
    edu_header = soup.find("h2", string=lambda t: t and "Education and training" in t)
    if edu_header:
        current_tag = edu_header.find_next_sibling()
        while current_tag and current_tag.name != 'h2':
            if current_tag.name == 'p' or current_tag.name == 'h4':
                cv_text_list.append(current_tag.text.strip())
            current_tag = current_tag.find_next_sibling()
            
    # --- 2. Fetch the Work experience section ---
    work_header = soup.find("h2", string=lambda t: t and "Work experience" in t)
    if work_header:
        current_tag = work_header.find_next_sibling()
        while current_tag and current_tag.name != 'h2':
            if current_tag.name == 'p' or current_tag.name == 'h4':
                cv_text_list.append(current_tag.text.strip())
            current_tag = current_tag.find_next_sibling()

    # --- 3. Fetch the Skills section ---
    skills_header = soup.find("h2", string=lambda t: t and "Skills" in t)
    if skills_header:
        
        # --- NEW UPDATE (FIX FOR THE ERROR) ---
        # Instead of 'find_next_sibling("ul")', we use the more flexible 'find_next("ul")'.
        # This finds the list even if there are other tags between <h2> and <ul>.
        skills_ul = skills_header.find_next("ul") 
        # ---------------------------------------------
        
        if skills_ul:
            skill_items = skills_ul.find_all("li")
            for item in skill_items:
                skill_text = item.text.strip()
                if skill_text: # Skip empty <li> tags
                    cv_text_list.append(skill_text)
                    cv_skill_list.append(skill_text)

    # --- 4. Fetch the Certificates section ---
    cert_header = soup.find("h2", string=lambda t: t and "Certificates" in t)
    if cert_header:
        current_tag = cert_header.find_next_sibling()
        while current_tag and current_tag.name != 'h2':
            if current_tag.name == 'p':
                cv_text_list.append(current_tag.text.strip())
            current_tag = current_tag.find_next_sibling()
            
    # --- 5. Merge the outputs ---
    cv_full_text = "\n".join(list(filter(None, cv_text_list))) # Remove empty lines
    
    df_cv_skills = pd.DataFrame(cv_skill_list, columns=["raw_skill_name"])
    df_cv_skills.to_pickle("IP1_output_candidate_raw_skills.pkl")
    
    with open("IP1_output_candidate_full_text.txt", "w", encoding="utf-8") as f:
        f.write(cv_full_text)

    print(f"\n--- IP 1 - Step 3 COMPLETED (SELENIUM/FIXED) ---")
    print("Europass CV (HTML) parsed successfully.")
    print(f"Found {len(cv_skill_list)} raw skill texts in total and saved them to 'IP1_output_candidate_raw_skills.pkl'.")
    print("The full text of the CV was saved to 'IP1_output_candidate_full_text.txt'.")

except requests.exceptions.RequestException as e:
    print(f"ERROR: Could not connect to the Europass link: {e}")
except Exception as e:
    print(f"An unknown error occurred: {e}")
finally:
    driver.quit()
    print("Browser closed.")

IP 1 - Step 3 started: fetching the Europass CV link with Selenium...
URL: https://europa.eu/europass/eportfolio/api/eprofile/shared-profile/example/12345678-1234-1234-1234-123456789012?view=html
Europass profile (JavaScript) loaded. Parsing data...

--- IP 1 - Step 3 COMPLETED (SELENIUM/FIXED) ---
Europass CV (HTML) parsed successfully.
Found 14 raw skill texts in total and saved them to 'IP1_output_candidate_raw_skills.pkl'.
The full text of the CV was saved to 'IP1_output_candidate_full_text.txt'.
Browser closed.
